# Operator Learning: DeepONet & FNO Hands-On

In this notebook we implement minimal versions of **DeepONet** and **Fourier Neural Operator (FNO)** and train them on a 1D diffusion equation.

## Problem Setup

We learn the solution operator for the 1D heat equation:

$$\frac{\partial u}{\partial t} = \nu \frac{\partial^2 u}{\partial x^2}, \quad u(x, 0) = u_0(x)$$

Given random initial conditions $u_0(x)$, the operator maps $u_0 \mapsto u(x, T)$ at a fixed time $T$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

# Grid parameters
N = 64          # spatial points
nu = 0.01       # diffusivity
T_final = 0.1   # evaluation time
x = np.linspace(0, 1, N, endpoint=False)
dx = x[1] - x[0]

### Data Generation

We generate random initial conditions as sums of Fourier modes with decaying amplitudes, then solve the heat equation exactly using the spectral method (FFT-based). This gives us ground-truth (input, output) pairs for training.

In [ ]:
def random_ic(n_samples, n_modes=5):
    """Generate random initial conditions as sum of Fourier modes."""
    u0 = np.zeros((n_samples, N))
    for k in range(1, n_modes + 1):
        a = np.random.randn(n_samples, 1) / k
        b = np.random.randn(n_samples, 1) / k
        u0 += a * np.sin(2 * np.pi * k * x) + b * np.cos(2 * np.pi * k * x)
    return u0

def exact_solution(u0, t, nu):
    """Solve heat equation spectrally (exact for periodic BC)."""
    u0_hat = np.fft.fft(u0, axis=-1)
    k = np.fft.fftfreq(N, d=dx) * 2 * np.pi
    decay = np.exp(-nu * k**2 * t)
    return np.real(np.fft.ifft(u0_hat * decay, axis=-1))

# Generate training and test data
n_train, n_test = 1000, 200
u0_train = random_ic(n_train)
uT_train = exact_solution(u0_train, T_final, nu)
u0_test = random_ic(n_test)
uT_test = exact_solution(u0_test, T_final, nu)

print(f'Training: {n_train} samples, Test: {n_test} samples')
print(f'Grid: {N} points, T = {T_final}, nu = {nu}')

## Part 1: DeepONet

The **branch network** takes the input function $u_0$ (sampled at sensor locations) and produces coefficients $b_1, \ldots, b_p$.

The **trunk network** takes query points $y$ and produces basis functions $t_1(y), \ldots, t_p(y)$.

$$\mathcal{G}(u_0)(y) = \sum_{i=1}^{p} b_i(u_0) \cdot t_i(y) = \langle \mathbf{b}(u_0), \mathbf{t}(y) \rangle$$

In [ ]:
class DeepONet(nn.Module):
    def __init__(self, u_dim, y_dim=1, p=64, hidden=128):
        super().__init__()
        self.branch = nn.Sequential(
            nn.Linear(u_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, p)
        )
        self.trunk = nn.Sequential(
            nn.Linear(y_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, p)
        )
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, u, y):
        # u: (batch, u_dim) — input function values
        # y: (batch, n_query, 1) — query points
        b = self.branch(u)          # (batch, p)
        t = self.trunk(y)           # (batch, n_query, p)
        out = torch.einsum('bp,bqp->bq', b, t)
        return out + self.bias

# Prepare data
u0_t = torch.tensor(u0_train, dtype=torch.float32)
uT_t = torch.tensor(uT_train, dtype=torch.float32)
y_grid = torch.tensor(x, dtype=torch.float32).unsqueeze(-1)  # (N, 1)
y_batch = y_grid.unsqueeze(0).expand(n_train, -1, -1)        # (n_train, N, 1)

model_don = DeepONet(u_dim=N, p=64)
optimizer = optim.Adam(model_don.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)

# Train
losses = []
for epoch in range(1500):
    pred = model_don(u0_t, y_batch)
    loss = nn.MSELoss()(pred, uT_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    losses.append(loss.item())
    if (epoch + 1) % 300 == 0:
        print(f'Epoch {epoch+1}: loss = {loss.item():.6f}')

plt.semilogy(losses)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss'); plt.title('DeepONet Training')
plt.grid(alpha=0.3); plt.show()

### DeepONet Architecture

The model below has two sub-networks:

- **Branch** (`nn.Sequential`): takes the full input function $u_0$ sampled at $N$ sensor points and outputs $p$ coefficients.
- **Trunk** (`nn.Sequential`): takes query coordinates $y$ and outputs $p$ basis function values.

The output is their dot product: $\mathcal{G}(u_0)(y) = \langle \mathbf{b}(u_0), \mathbf{t}(y) \rangle + \text{bias}$.

In [ ]:
# Test DeepONet
u0_te = torch.tensor(u0_test, dtype=torch.float32)
y_te = y_grid.unsqueeze(0).expand(n_test, -1, -1)
with torch.no_grad():
    pred_don = model_don(u0_te, y_te).numpy()

rel_err = np.mean(np.linalg.norm(pred_don - uT_test, axis=1) / (np.linalg.norm(uT_test, axis=1) + 1e-8))
print(f'DeepONet relative L2 error: {rel_err:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for i, ax in enumerate(axes):
    ax.plot(x, uT_test[i], 'k-', label='True', linewidth=2)
    ax.plot(x, pred_don[i], '--', color='#61afef', label='DeepONet')
    ax.set_title(f'Test {i+1}', fontsize=11)
    ax.legend(fontsize=9); ax.set_xlabel('x')
plt.suptitle('DeepONet Predictions on Test Set', fontsize=13)
plt.tight_layout(); plt.show()

## Part 2: Fourier Neural Operator (FNO)

Each FNO layer applies: $v^{(\ell+1)} = \sigma\big(W v^{(\ell)} + \mathcal{F}^{-1}(R_\phi \cdot \mathcal{F}(v^{(\ell)}))\big)$

- FFT the input, multiply low modes by learned weights $R_\phi$, iFFT back
- Add a local linear transform $W v$ (skip connection in spatial domain)
- Apply nonlinear activation

### FNO Architecture

The key building block is `SpectralConv1d`: it applies the FFT, multiplies the lowest `modes` Fourier coefficients by a learned complex weight matrix $R_\phi$, then applies the inverse FFT.

The full `FNO1d` model stacks multiple such layers. Each layer also has a local 1x1 convolution $W$ (skip connection in spatial domain). The input is first "lifted" from 1 channel to `width` channels, and the output is projected back to 1 channel.

In [ ]:
class SpectralConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, modes):
        super().__init__()
        self.modes = modes
        scale = 1 / (in_ch * out_ch)
        self.weights = nn.Parameter(scale * torch.randn(in_ch, out_ch, modes, dtype=torch.cfloat))

    def forward(self, x):
        # x: (batch, in_ch, N)
        x_ft = torch.fft.rfft(x)
        out_ft = torch.zeros(x.size(0), self.weights.size(1), x_ft.size(-1),
                             dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes] = torch.einsum('bix,iox->box',
                                                  x_ft[:, :, :self.modes], self.weights)
        return torch.fft.irfft(out_ft, n=x.size(-1))

class FNO1d(nn.Module):
    def __init__(self, modes=16, width=32, n_layers=4):
        super().__init__()
        self.lift = nn.Linear(1, width)   # lift input to channel dim
        self.layers = nn.ModuleList()
        self.ws = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(SpectralConv1d(width, width, modes))
            self.ws.append(nn.Conv1d(width, width, 1))
        self.proj = nn.Linear(width, 1)   # project back to 1D

    def forward(self, x):
        # x: (batch, N) → output: (batch, N)
        x = x.unsqueeze(-1)               # (batch, N, 1)
        x = self.lift(x)                   # (batch, N, width)
        x = x.permute(0, 2, 1)            # (batch, width, N)
        for conv, w in zip(self.layers, self.ws):
            x = torch.relu(conv(x) + w(x))
        x = x.permute(0, 2, 1)            # (batch, N, width)
        return self.proj(x).squeeze(-1)    # (batch, N)

model_fno = FNO1d(modes=16, width=32, n_layers=4)
optimizer = optim.Adam(model_fno.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)

losses_fno = []
for epoch in range(1500):
    pred = model_fno(u0_t)
    loss = nn.MSELoss()(pred, uT_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    losses_fno.append(loss.item())
    if (epoch + 1) % 300 == 0:
        print(f'Epoch {epoch+1}: loss = {loss.item():.6f}')

plt.semilogy(losses_fno)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss'); plt.title('FNO Training')
plt.grid(alpha=0.3); plt.show()

In [ ]:
# Test FNO
with torch.no_grad():
    pred_fno = model_fno(u0_te).numpy()

rel_err_fno = np.mean(np.linalg.norm(pred_fno - uT_test, axis=1) / (np.linalg.norm(uT_test, axis=1) + 1e-8))
print(f'FNO relative L2 error: {rel_err_fno:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for i, ax in enumerate(axes):
    ax.plot(x, uT_test[i], 'k-', label='True', linewidth=2)
    ax.plot(x, pred_fno[i], '--', color='#c678dd', label='FNO')
    ax.set_title(f'Test {i+1}', fontsize=11)
    ax.legend(fontsize=9); ax.set_xlabel('x')
plt.suptitle('FNO Predictions on Test Set', fontsize=13)
plt.tight_layout(); plt.show()

## Comparison

Let's compare DeepONet and FNO side by side on the same test samples.

In [ ]:
print(f'DeepONet relative L2 error: {rel_err:.4f}')
print(f'FNO      relative L2 error: {rel_err_fno:.4f}')

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for i in range(3):
    axes[0, i].plot(x, uT_test[i], 'k-', lw=2, label='True')
    axes[0, i].plot(x, pred_don[i], '--', color='#61afef', label='DeepONet')
    axes[0, i].set_title(f'Test {i+1}'); axes[0, i].legend(fontsize=9)
    axes[1, i].plot(x, uT_test[i], 'k-', lw=2, label='True')
    axes[1, i].plot(x, pred_fno[i], '--', color='#c678dd', label='FNO')
    axes[1, i].set_xlabel('x'); axes[1, i].legend(fontsize=9)
axes[0, 0].set_ylabel('DeepONet'); axes[1, 0].set_ylabel('FNO')
plt.suptitle('DeepONet vs FNO on Heat Equation', fontsize=14)
plt.tight_layout(); plt.show()

## Exercises

1. **Resolution transfer (FNO)**: Train the FNO on a 64-point grid, then evaluate it on a 128-point grid. Does it transfer? Why?

2. **Sensor sparsity (DeepONet)**: Reduce the number of sensor points in the branch network (e.g., use every 4th point). How does accuracy degrade?

3. **Harder PDE**: Replace the heat equation with Burgers' equation $u_t + u u_x = \nu u_{xx}$. Which method handles the nonlinearity better?

4. **Physics-informed DeepONet**: Add a PDE residual loss term $\|\mathcal{N}[\mathcal{G}_\theta(u_0)] - 0\|^2$ and train with fewer data pairs. Does it help?